In [0]:
%run ../../config/config

In [0]:
%run  ../../config/sqlconfig

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, FloatType, TimestampType
from pyspark.sql.functions import col, lit, udf, to_timestamp, to_date, to_utc_timestamp, from_utc_timestamp, unix_timestamp, from_unixtime, datediff, months_between, round, dayofmonth, dayofweek, dayofyear, month, year, hour, minute, second, weekofyear, date_format, current_timestamp, current_date, date_add, date_sub, date_trunc, date_diff

In [0]:
class SilverCustomerStreamETL:
    def __init__(self, spark):
        self.spark = spark
        self.catalog = CONFIG['catalog']['name']
        self.schema_name = CONFIG['catalog']['schema']
        
        # up-stream table name
        self.upstream_table_name = TABLES['bronze']['customer']

        # down-stream table name
        self.downstream_table_name = TABLES['silver']['customer']

        #path
        self.checkpoint_path = CONFIG['path']['checkpoint']

    def read_stream(self):
        df = self.spark.readStream.table(f"{self.catalog}.{self.schema_name}.{self.upstream_table_name }")
        return df

    def column_to_keep(self, df):
        return df.select( "customer_key", "customer_name", "gender", "city", "surrogate_key")
    
    def change_data_type(self, df):
        df_casted = (
            df.withColumn("customer_key", col("customer_key").cast(IntegerType()))
            .withColumn("customer_name", col("customer_name").cast(StringType()))
            .withColumn("gender", col("gender").cast(StringType()))
            .withColumn("city", col("city").cast(StringType()))
        )
        return df_casted
    
    # ----------------------------------------------------
    # Upsert to silver
    # ----------------------------------------------------
    def upsert_to_silver(self, microBatchDF, batchId):
        try:
            dedup_df = microBatchDF.dropDuplicates(['surrogate_key'])
            dedup_df.createOrReplaceTempView("v_dedup_customer")
            microBatchDF.sparkSession.sql(f"""
                MERGE INTO {self.catalog}.{self.schema_name}.{self.downstream_table_name} t
                USING v_dedup_customer s
                ON t.surrogate_key = s.surrogate_key
                WHEN NOT MATCHED
                    THEN INSERT *
            """)            
        except Exception as e:
            print(f"Error: upsert_to_silver : {e}")

    # --------------------------------
    # Only for droping table
    # ---------------------------------
    #def drop_table(self):
    #    dbutils.fs.rm(f"{self.checkpoint_path}/{self.downstream_table_name}", True)

    # -------------------------------
    # Write to Delta Table 
    # -------------------------------

    def write_stream(self, df):
        try:
            query= (
                df.writeStream
                .option("checkpointLocation", f"{self.checkpoint_path}/{self.downstream_table_name}")
                .foreachBatch(self.upsert_to_silver)
                .trigger(availableNow=True)
                .table(f"{self.catalog}.{self.schema_name}.{self.downstream_table_name}")
            )
            query.awaitTermination()
        except Exception as e:
            print(f"Error: write_stream {self.downstream_table_name}: {e}")

    def run(self):
        df = self.read_stream()
        df = self.column_to_keep(df)
        df_casted = self.change_data_type(df)
        self.write_stream(df_casted)
        print("success full run")


In [0]:
obj = SilverCustomerStreamETL(spark)
obj.run()